#CÀI ĐẶT THƯ VIỆN

In [41]:
!pip install transformers[sentencepiece] rouge-score sacremoses

In [42]:
import json
import torch
import os
import re
import unicodedata
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from tqdm import tqdm
from datasets import load_dataset

In [43]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using device:', device)

using device: cuda


#CÁC HÀM TIỆN ÍCH (UTILS)

In [44]:
def clean_text(text: str) -> str:
    """
    Làm sạch văn bản tiếng Việt:
    - Bỏ HTML tags
    - Chuẩn hóa Unicode (NFC)
    - Thay '_' -> ' '
    - Xóa newline
    - Chuẩn hóa khoảng trắng
    - Chuẩn hóa dấu câu (có khoảng trắng sau dấu)
    """

    if not isinstance(text, str):
        return ""

    # Bỏ HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # Chuẩn hóa Unicode (NFC)
    text = unicodedata.normalize('NFC', text)

    # thay underscore thành space
    text = text.replace('_', ' ')

    # Xóa newline
    text = text.replace('\n', ' ')

    #chuẩn hóa dấu câu
    text = re.sub(r'([.,!?;:])', r' \1 ', text)

    #chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def truncate_text(text: str, max_words: int = 512) -> str:
    """
    Cắt văn bản thành một đoạn có tối đa `max_words` từ.
    """
    if not isinstance(text, str):
        return ""

    words = text.split()
    if len(words) <= max_words:
        return text
    else:
        return ' '.join(words[:max_words]) + "..."


def compute_length(text: str) -> int:
    """
    Tính số lượng từ trong văn bản.
    """
    if not isinstance(text, str):
        return 0

    words = text.split()
    return len(words)


def compute_compression_ratio(article: str, summary: str) -> float:
    """
    Tính tỷ lệ nén giữa văn bản gốc và văn bản đã được làm sạch.
    """
    article_len = compute_length(article)
    summary_len = compute_length(summary)

    if article_len == 0:
        return 0.0

    return summary_len / article_len


def is_abnormal_sample(article: str, summary: str) -> bool:
    """
    Detect sample bất thường:
    - article quá ngắn
    - ratio quá cao
    """
    article_len = compute_length(article)
    ratio = compute_compression_ratio(article, summary)

    if article_len < 50 or ratio > 0.8:
        return True
    return False


# TẠO DỮ LIỆU KIỂM THỬ (TEST SET)

In [45]:
def is_valid_sample(article: str, summary: str) -> bool:
    """
    Filter sample:
    - article đủ dài
    - ratio không quá cao (loại sample lỗi)
    """
    if not article or not summary:
        return False

    article_len = len(article.split())
    ratio = compute_compression_ratio(article, summary)

    # rule
    if article_len < 20:
        return False

    if ratio > 0.3:
        return False

    return True

def run_create_test_set(num_samples=300, seed=42):
    print("--- Loading dataset Vietnews... ---")
    dataset = load_dataset("harouzie/vietnews", split='test')
    raw_test = dataset.shuffle(seed=seed)

    test_data = []
    print("--- Creating test set... ---")
    for sample in raw_test:
        article = clean_text(sample["article"])
        summary = clean_text(sample["abstract"])

        if not is_valid_sample(article, summary):
            continue

        test_data.append({"article": article, "abstract": summary})
        if len(test_data) >= num_samples:
            break

    print(f"Collected {len(test_data)} samples")
    return test_data


#HÀM ĐÁNH GIÁ MODEL (EVALUATE)

In [46]:
def resolve_model_path_colab(model_path: str) -> str:
    # Nếu là Repo ID trên Hugging Face (có dấu /) thì trả về
    if "/" in model_path and not os.path.exists(model_path):
        return model_path

    if not os.path.exists(model_path):
        raise ValueError(f"Path không tồn tại: {model_path}")

    files = os.listdir(model_path)
    if "config.json" in files:
        return model_path
    checkpoints = [f for f in files if f.startswith("checkpoint")]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split('-')[-1]) if '-' in x else x)
        return os.path.join(model_path, checkpoints[-1])
    return model_path

def evaluate_model_colab(model_path: str, test_data: list, output_path: str = "rouge_results.json"):
    # Giải quyết đường dẫn
    model_path = resolve_model_path_colab(model_path)
    print(f"Loading model from: {model_path}")

    # Load model & tokenizer
    # Colab có thể tải từ Hugging Face
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    print(f"Device: {device}")

    # ROUGE setup
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    results = {'rouge1': [], 'rouge2': [], 'rougeL': [], 'details': []}

    print("Evaluating...")
    for item in tqdm(test_data):
        article = clean_text(item['article'])
        reference = clean_text(item['abstract'])

        inputs = tokenizer(article, max_length=512, truncation=True, return_tensors="pt").to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                inputs["input_ids"],
                max_length=150,
                min_length=30,
                num_beams=4,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )

        generated = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        generated = clean_text(generated)

        # Tính ROUGE
        scores = scorer.score(reference, generated)
        r1, r2, rl = scores['rouge1'].fmeasure, scores['rouge2'].fmeasure, scores['rougeL'].fmeasure

        results['rouge1'].append(r1)
        results['rouge2'].append(r2)
        results['rougeL'].append(rl)
        results['details'].append({
            "reference": reference,
            "generated": generated,
            "rouge1": round(r1, 4), "rouge2": round(r2, 4), "rougeL": round(rl, 4)
        })

    # Tính trung bình
    avg_r1 = sum(results['rouge1']) / len(results['rouge1'])
    avg_r2 = sum(results['rouge2']) / len(results['rouge2'])
    avg_rl = sum(results['rougeL']) / len(results['rougeL'])

    print("\n" + "="*30)
    print("ROUGE RESULTS")
    print(f"ROUGE-1: {avg_r1:.4f}")
    print(f"ROUGE-2: {avg_r2:.4f}")
    print(f"ROUGE-L: {avg_rl:.4f}")
    print("="*30)

    # Lưu kết quả
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump({
            "model": model_path,
            "average": {"rouge1": round(avg_r1, 4), "rouge2": round(avg_r2, 4), "rougeL": round(avg_rl, 4)},
            "details": results['details']
        }, f, ensure_ascii=False, indent=2)
    print(f"Saved results to {output_path}")

#tóm tắt lỗi

In [51]:
import json
import os

def create_error_summary(input_path="rouge_results.json", output_path="error_analysis.txt", top_n=10):
    """
    Phân tích kết quả ROUGE để tìm ra các mẫu model làm tệ nhất và tốt nhất.
    """
    if not os.path.exists(input_path):
        print(f"Không tìm thấy file {input_path}. Hãy chạy evaluate.py trước!")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    details = data.get("details", [])
    if not details:
        print("File kết quả không có dữ liệu chi tiết (details).")
        return

    # 1. Sắp xếp danh sách theo điểm ROUGE-L tăng dần (tệ nhất lên đầu)
    worst_samples = sorted(details, key=lambda x: x["rougeL"])

    # 2. Lấy danh sách tốt nhất (để đối chiếu)
    best_samples = sorted(details, key=lambda x: x["rougeL"], reverse=True)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("==================================================\n")
        f.write("         BÁO CÁO PHÂN TÍCH LỖI TÓM TẮT            \n")
        f.write("==================================================\n\n")

        f.write(f"Model: {data.get('model', 'Unknown')}\n")
        f.write(f"Điểm ROUGE trung bình:\n")
        avg = data.get("average", {})
        f.write(f" - ROUGE-1: {avg.get('rouge1'):.4f}\n")
        f.write(f" - ROUGE-2: {avg.get('rouge2'):.4f}\n")
        f.write(f" - ROUGE-L: {avg.get('rougeL'):.4f}\n\n")

        # PHẦN 1: TOP CÁC CÂU TỆ NHẤT
        f.write(f"--- TOP {top_n} MẪU CÓ ĐIỂM THẤP NHẤT (CẦN PHÂN TÍCH LỖI) ---\n")
        for i, sample in enumerate(worst_samples[:top_n]):
            f.write(f"\n[Mẫu lỗi số {i+1}]\n")
            f.write(f" > ROUGE-L: {sample['rougeL']:.4f} | ROUGE-1: {sample['rouge1']:.4f}\n")
            f.write(f" > Tham chiếu (Gold): {sample['reference']}\n")
            f.write(f" > Model viết: {sample['generated']}\n")
            f.write("-" * 30 + "\n")

        f.write("\n" + "="*50 + "\n\n")

        # PHẦN 2: TOP CÁC CÂU TỐT NHẤT
        f.write(f"--- TOP {top_n} MẪU CÓ ĐIỂM CAO NHẤT (MODEL LÀM TỐT) ---\n")
        for i, sample in enumerate(best_samples[:top_n]):
            f.write(f"\n[Mẫu tốt số {i+1}]\n")
            f.write(f" > ROUGE-L: {sample['rougeL']:.4f} | ROUGE-1: {sample['rouge1']:.4f}\n")
            f.write(f" > Tham chiếu (Gold): {sample['reference']}\n")
            f.write(f" > Model viết: {sample['generated']}\n")
            f.write("-" * 30 + "\n")

    print(f"Đã tạo báo cáo phân tích lỗi tại: {output_path}")


In [50]:
if __name__ == "__main__":
    # Bước 1: Tạo dữ liệu test (dùng hàm của ông)
    test_samples = run_create_test_set(num_samples=300)

    # Bước 2: Chạy đánh giá với Model trên Hugging Face
    MODEL_ID = "OrdinaryAI/visum-model" # Tên repo của ông
    evaluate_model_colab(MODEL_ID, test_samples)

    create_error_summary()

--- Loading dataset Vietnews... ---
--- Creating test set... ---
Collected 300 samples
Loading model from: OrdinaryAI/visum-model


Loading weights:   0%|          | 0/518 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Device: cuda
Evaluating...


100%|██████████| 300/300 [04:50<00:00,  1.03it/s]


ROUGE RESULTS
ROUGE-1: 0.5851
ROUGE-2: 0.2686
ROUGE-L: 0.3713
Saved results to rouge_results.json
Đã tạo báo cáo phân tích lỗi tại: error_analysis.txt


- ROUGE-1 (0.5851): Rất cao (trên 58%). Điều này cho thấy model cực kỳ giỏi trong việc bắt được các "keywords" (từ đơn) quan trọng từ bài báo gốc.

- ROUGE-2 (0.2686): Khá tốt (gần 27%). Chỉ số này đo lường sự mượt mà của các cụm từ. Với mức này, câu văn tóm tắt của AI có độ trôi chảy cao, giống cách hành văn của con người.

- ROUGE-L (0.3713): Rất ổn (37%). Nó cho thấy cấu trúc câu và thứ tự ý chính được giữ vững so với bản gốc.

=> Kết quả thực nghiệm trên 300 mẫu dữ liệu VietNews cho thấy hệ thống đạt hiệu năng vượt trội với điểm ROUGE-1 là 0.5851. Điều này chứng minh quy trình tiền xử lý dữ liệu và cấu hình model BARTpho đã tối ưu hóa được việc giữ lại thông tin quan trọng trong khi vẫn đảm bảo tính súc tích của bản tóm tắt.